# 03 · TRUST and REINFORCE: voting, confidence, abstention and expert iteration

Needs the fine-tuned adapter from notebook 02 (the newest checkpoint is used if training stopped midway).

**Part A (Steps 5.4–5.7):** sample K answers per question, vote with weights from the verifier, turn the vote into a confidence score, and choose a threshold that keeps the error rate on *answered* questions below a target.

**Part B (Steps 4.2, 4.5):** expert iteration. Sample answers on training questions, keep only those that are correct **and** cite the right evidence (the evidence-aware reward), and fine-tune on them.

Rough T4 timing: Part A ≈ 40–60 min, Part B ≈ 1.5–2 h. Every step saves to Drive and resumes after a disconnect.

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
from finsight.infer import load_model, predict
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def latest_adapter(run):
    """The finished adapter, or the newest checkpoint if training stopped midway."""
    import glob
    if os.path.exists(f"{run}/final"):
        return f"{run}/final"
    ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    assert ckpts, f"No adapter in {run} - run notebook 02 (training) first"
    print("NOTE: training has not finished; using", ckpts[-1])
    return ckpts[-1]
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SFT_RUN = f"{OUT}/sft_Qwen2.5-3B-Instruct"
ADAPTER = latest_adapter(SFT_RUN)
model, tok = load_model(MODEL_ID, adapter=ADAPTER)

## Part A · Greedy vs. verification-weighted voting (dev sample)

In [ ]:
from finsight.infer import sample as draw_samples
from finsight.trust import vote, risk_coverage, choose_threshold, selective_report
from finsight.reinforce import answer_correct, evidence_f1

N, K = 500, 5
eval_qs = sample(q_dev, N, seed=1)
greedy = predict(model, tok, docs_dev, eval_qs, f"{SFT_RUN}/greedy_dev{N}.jsonl", batch_size=8)
print_report(score(eval_qs, greedy), "fine-tuned, greedy")

raw = draw_samples(model, tok, docs_dev, eval_qs, f"{SFT_RUN}/samples_dev{N}_k{K}.jsonl", k=K, batch_size=2)
voted = {q["uid"]: vote(raw[q["uid"]], q, docs_dev[q["doc_uid"]]) for q in eval_qs}
print_report(score(eval_qs, voted), f"fine-tuned, verification-weighted vote (k={K})")

## Confidence and selective answering
The threshold is chosen on one half of the sample and checked on the other half, so the reported error rate is honest.

In [ ]:
import matplotlib.pyplot as plt
conf = [voted[q["uid"]]["confidence"] for q in eval_qs]
ok = [int(answer_correct(voted[q["uid"]]["answer"], voted[q["uid"]]["scale"], q)) for q in eval_qs]
half = len(eval_qs) // 2
for target in (0.05, 0.10, 0.20):
    t = choose_threshold(conf[:half], ok[:half], target)
    print(f"target error <= {target:.0%}:", selective_report(conf[half:], ok[half:], t) if t is not None else "not reachable on this sample")

rc = risk_coverage(conf, ok)
plt.figure(figsize=(6, 4))
plt.plot([c * 100 for c, _, _ in rc], [a * 100 for _, a, _ in rc], marker=".")
plt.xlabel("coverage: % of questions answered"); plt.ylabel("accuracy on answered questions (%)")
plt.title("Risk-coverage curve (dev sample)"); plt.grid(alpha=.3)
plt.savefig(f"{SFT_RUN}/risk_coverage.png", dpi=150, bbox_inches="tight"); plt.show()

## Part B · Expert iteration (light-weight reinforcement learning)
Keeps only samples that are correct, cite evidence with F1 ≥ 0.5 and whose numbers appear in the cited evidence. Correct answers built on the wrong evidence (*right for the wrong reason*) are counted and thrown away.

In [ ]:
import json
from finsight.reinforce import select_targets

N_TRAIN, K_TRAIN = 1000, 4
ei_qs = sample(q_train, N_TRAIN, seed=2)
raw_tr = draw_samples(model, tok, docs_train, ei_qs, f"{SFT_RUN}/samples_train{N_TRAIN}_k{K_TRAIN}.jsonl",
                      k=K_TRAIN, batch_size=2)
targets, stats = select_targets(ei_qs, docs_train, raw_tr)
print(stats)
json.dump(targets, open(f"{SFT_RUN}/ei_targets.json", "w"))

In [ ]:
import gc
from finsight.train import train
del model; gc.collect(); torch.cuda.empty_cache()

EI_RUN = f"{OUT}/ei1_Qwen2.5-3B-Instruct"
ei_adapter = train(MODEL_ID, docs_train, ei_qs, EI_RUN, epochs=1, lr=5e-5, batch_size=1, grad_accum=8,
                   targets=targets, init_adapter=ADAPTER, save_steps=50)

In [ ]:
model, tok = load_model(MODEL_ID, adapter=ei_adapter)
greedy_ei = predict(model, tok, docs_dev, eval_qs, f"{EI_RUN}/greedy_dev{N}.jsonl", batch_size=8)
print_report(score(eval_qs, greedy), "SFT, greedy")
print_report(score(eval_qs, greedy_ei), "SFT + expert iteration, greedy")

def wrong_reason_rate(preds):
    right = [q for q in eval_qs if answer_correct(preds[q["uid"]]["answer"], preds[q["uid"]]["scale"], q)]
    bad = [q for q in right if evidence_f1(preds[q["uid"]]["evidence"], q["evidence"]) < 0.5]
    return 100 * len(bad) / max(1, len(right))
print(f"right-for-the-wrong-reason: SFT {wrong_reason_rate(greedy):.1f}% -> SFT+EI {wrong_reason_rate(greedy_ei):.1f}% of correct answers")